# NeuroDiffusion: A100 Colab Training Launchpad

This notebook handles the end-to-end pipeline: MAE Pre-training -> LDM Fine-tuning -> Image Generation.

### Pre-requisites:
1. Ensure you have `v1-5-pruned.ckpt`, `eeg_5_95_std.pth`, and `block_splits_by_image_single.pth` uploaded to a folder in your Google Drive named `NeuroDiffusion_Data`.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Setup Repository

In [ ]:
%cd /content
!rm -rf NeuroDiffusion
!git clone https://github.com/JoaoLucasVeras/NeuroDiffusion.git
%cd NeuroDiffusion
!git checkout hpc-dev

## 3. Install Dependencies

In [ ]:
!pip install einops omegaconf kornia torch-fidelity timm wandb torchmetrics natsort h5py mne transformers

## 4. Link & Unpack Datasets
Stage 1 requires `.npy` files. We unpack them from your `.pth` file to save upload time.

In [ ]:
import os
import torch
import numpy as np

# Create directories
!mkdir -p datasets
!mkdir -p pretrains/models

# Symlink your Drive data to the local project folder
DRIVE_DATA = '/content/drive/MyDrive/NeuroDiffusion_Data'
!ln -sf "{DRIVE_DATA}/eeg_5_95_std.pth" datasets/eeg_5_95_std.pth
!ln -sf "{DRIVE_DATA}/block_splits_by_image_single.pth" datasets/block_splits_by_image_single.pth
!ln -sf "{DRIVE_DATA}/v1-5-pruned.ckpt" pretrains/models/v1-5-pruned.ckpt

print("📦 Unpacking data for Stage 1...")
os.makedirs('datasets/mne_data', exist_ok=True)
loaded = torch.load('datasets/eeg_5_95_std.pth')
for i, item in enumerate(loaded['dataset']):
    np.save(f'datasets/mne_data/sub001_chunk_{i:03d}.npy', item['eeg'].numpy())

print("✅ Data Ready")

## 5. Stage 1: MAE Pre-training
Run this to train the EEG encoder. Results are saved in `results/eeg_pretrain/`.

In [ ]:
%cd /content/NeuroDiffusion/code
!python stageA1_eeg_pretrain.py --batch_size 128 --num_epoch 500

## 6. Stage 2: Diffusion Fine-tuning
**Note**: Replace the path below with your actual MAE checkpoint from Stage 1.

In [ ]:
MAE_CHECKPOINT = "../results/eeg_pretrain/LATEST_RUN/checkpoints/checkpoint.pth" # Update this

%cd /content/NeuroDiffusion/code
!python eeg_ldm.py --pretrain_mbm_path {MAE_CHECKPOINT} --batch_size 16

## 7. Stage 3: Image Generation (Inference)
Transform imagination brainwaves into images.

In [ ]:
STAGE2_CHECKPOINT = "../exps/eeg_ldm/LATEST_RUN/checkpoints/last.ckpt" # Update this

%cd /content/NeuroDiffusion/code
!python gen_eval_eeg.py --dataset EEG --model_path {STAGE2_CHECKPOINT}